# Business KPIs - Frigus

KPIs de negocio para o projeto Frigus: usuarios, grupos, planos, transacoes, produtos, compras, receitas, desperdicio e conversas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time

BRONZE_SCHEMA = "frigus.data_engineering_bronze"
SILVER_SCHEMA = "frigus.data_engineering_silver"
METRICS_SCHEMA = "frigus.data_engineering_metrics"

execution_timestamp = datetime.now()

In [0]:
# ============================================================
# BUSINESS KPIs - USUARIOS, GRUPOS, PLANOS E TRANSACOES
# ============================================================

BRONZE = "frigus.data_engineering_bronze"
SILVER = "frigus.data_engineering_silver"

print("=" * 60)
print("KPIs DE USUARIOS E GRUPOS")
print("=" * 60)

# Quantidade de usuarios (ativos, nao deletados)
total_usuarios = spark.sql(f"""
    SELECT COUNT(*) as total
    FROM {BRONZE}.users
    WHERE deleted_at IS NULL
""").collect()[0]["total"]
print(f"\nTotal de usuarios: {total_usuarios}")

# Usuarios por grupo
print("\n--- Usuarios por grupo ---")
spark.sql(f"""
    SELECT g.name as grupo, COUNT(ug.user_id) as qtd_usuarios
    FROM {BRONZE}.groups g
    LEFT JOIN {BRONZE}.user_groups ug ON g.id = ug.group_id
    WHERE g.deleted_at IS NULL
    GROUP BY g.id, g.name
    ORDER BY qtd_usuarios DESC
""").show(truncate=False)

# Media de usuarios por grupo
media_usuarios_grupo = spark.sql(f"""
    SELECT AVG(user_count) as media
    FROM (
        SELECT g.id, COUNT(ug.user_id) as user_count
        FROM {BRONZE}.groups g
        LEFT JOIN {BRONZE}.user_groups ug ON g.id = ug.group_id
        WHERE g.deleted_at IS NULL
        GROUP BY g.id
    )
""").collect()[0]["media"]
print(f"Media de usuarios por grupo: {media_usuarios_grupo:.2f}")

# Quantidade de usuarios por plano
print("\n--- Usuarios por plano ---")
spark.sql(f"""
    SELECT p.name as plano, COUNT(DISTINCT s.user_id) as qtd_usuarios
    FROM {SILVER}.plans p
    LEFT JOIN {BRONZE}.subscriptions s ON p.id = s.plan_id AND s.deleted_at IS NULL
    WHERE p.deleted_at IS NULL
    GROUP BY p.id, p.name
    ORDER BY qtd_usuarios DESC
""").show(truncate=False)

print("\n" + "=" * 60)
print("KPIs DE TRANSACOES")
print("=" * 60)

# Metodo de pagamento mais usado
print("\n--- Metodo de pagamento mais usado ---")
spark.sql(f"""
    SELECT payment_method, COUNT(*) as qtd
    FROM {BRONZE}.transactions
    GROUP BY payment_method
    ORDER BY qtd DESC
""").show(truncate=False)

# Media do amount em transacoes
media_amount = spark.sql(f"""
    SELECT AVG(amount) as media_amount
    FROM {BRONZE}.transactions
""").collect()[0]["media_amount"]
print(f"Media do amount em transacoes: {media_amount:.2f}")

# Taxa de retry em transacoes
print("\n--- Taxa de retry em transacoes ---")
spark.sql(f"""
    SELECT 
        COUNT(*) as total_transacoes,
        SUM(CASE WHEN attempts > 1 THEN 1 ELSE 0 END) as transacoes_com_retry,
        ROUND(SUM(CASE WHEN attempts > 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as taxa_retry_pct
    FROM {BRONZE}.transactions
""").show()

In [0]:
# ============================================================
# BUSINESS KPIs - PRODUTOS, COMPRAS, RECEITAS, DESPERDICIO E CONVERSAS
# ============================================================

BRONZE = "frigus.data_engineering_bronze"
SILVER = "frigus.data_engineering_silver"

print("=" * 60)
print("KPIs DE PRODUTOS E COMPRAS")
print("=" * 60)

# 1. Produtos mais presentes em contas (estoque)
print("\n--- Produtos mais presentes em estoque (top 10) ---")
spark.sql(f"""
    SELECT p.name as produto, SUM(sp.quantity) as qtd_total
    FROM {BRONZE}.stock_products sp
    JOIN {BRONZE}.products p ON sp.product_id = p.id
    GROUP BY p.id, p.name
    ORDER BY qtd_total DESC
    LIMIT 10
""").show(truncate=False)

# 2. Produtos mais colocados na lista de compras
print("\n--- Produtos mais colocados na lista de compras (top 10) ---")
spark.sql(f"""
    SELECT p.name as produto, COUNT(slp.id) as vezes_adicionado
    FROM {SILVER}.shopping_list_products slp
    JOIN {BRONZE}.products p ON slp.product_id = p.id
    GROUP BY p.id, p.name
    ORDER BY vezes_adicionado DESC
    LIMIT 10
""").show(truncate=False)

# 3. Receitas mais sugeridas (proxy: ingredientes que mais aparecem em listas de compras)
print("\n--- Receitas mais sugeridas (top 10) ---")
spark.sql(f"""
    WITH recipe_matches AS (
        SELECT r.id, r.name, COUNT(DISTINCT slp.id) as vezes_sugerida
        FROM {SILVER}.recipes r
        JOIN {SILVER}.recipe_ingredients ri ON r.id = ri.recipe_id
        JOIN {SILVER}.shopping_list_products slp ON ri.product_id = slp.product_id
        WHERE r.active = true
        GROUP BY r.id, r.name
    )
    SELECT name as receita, vezes_sugerida
    FROM recipe_matches
    ORDER BY vezes_sugerida DESC
    LIMIT 10
""").show(truncate=False)

print("\n" + "=" * 60)
print("KPIs DE DESPERDICIO")
print("=" * 60)

# 4. Perdas: quantidade, custo e taxa
print("\n--- Desperdicio: quantidade, custo e taxa ---")
spark.sql(f"""
    SELECT 
        SUM(sp.quantity) as quantidade_perdida,
        ROUND(SUM(sp.quantity * p.unit_price), 2) as custo_perdas,
        COUNT(d.id) as total_descartes
    FROM {SILVER}.discard d
    JOIN {BRONZE}.stock_products sp ON d.stock_product_id = sp.id
    JOIN {BRONZE}.products p ON sp.product_id = p.id
""").show(truncate=False)

taxa_perda = spark.sql(f"""
    SELECT 
        (SELECT COUNT(*) FROM {SILVER}.discard) as descartes,
        (SELECT COUNT(*) FROM {BRONZE}.stock_products) as total_estoque
""").collect()[0]
print(f"Taxa de perda: {taxa_perda['descartes']} descartes / {taxa_perda['total_estoque']} itens em estoque = {taxa_perda['descartes'] * 100.0 / max(1, taxa_perda['total_estoque']):.2f}%")

# 5. Distribuicao de motivos de descarte
print("\n--- Motivos de descarte ---")
spark.sql(f"""
    SELECT reason as motivo, COUNT(*) as qtd
    FROM {SILVER}.discard
    GROUP BY reason
    ORDER BY qtd DESC
""").show(truncate=False)

print("\n" + "=" * 60)
print("KPIs DE CONVERSAS")
print("=" * 60)

# 6. Numero medio de conversas por grupo
media_conversas = spark.sql(f"""
    SELECT AVG(conv_count) as media
    FROM (
        SELECT g.id, COUNT(c.id) as conv_count
        FROM {BRONZE}.groups g
        LEFT JOIN {BRONZE}.conversations c ON g.id = c.group_id
        WHERE g.deleted_at IS NULL
        GROUP BY g.id
    )
""").collect()[0]["media"]
print(f"\nMedia de conversas por grupo: {media_conversas:.2f}")

# 7. Total de conversas e por tipo
print("\n--- Conversas por tipo ---")
spark.sql(f"""
    SELECT conversation_type as tipo, COUNT(*) as qtd
    FROM {BRONZE}.conversations
    GROUP BY conversation_type
    ORDER BY qtd DESC
""").show(truncate=False)